In [ ]:
# Steam next-genre EDA (Jacob). Data: Kaggle "Steam Games Dataset 2025" (steam_dataset_2025_csv).
# Tables used: applications.csv (game metadata), application_genres.csv (appid, genre_id),
# genres.csv (id, name; one genre appears once per store language), reviews.csv (user, game, time).
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The raw CSVs are too big for GitHub (100 MB/file limit; reviews.csv ~670 MB), and the repo
# .gitignore already skips steam_dataset_2025_csv/. Everyone downloads it from Kaggle and puts the
# folder anywhere below. The first match wins.
CANDIDATES = [
    Path("steam_dataset_2025_csv"), Path("data/steam_dataset_2025_csv"),
    Path("../steam_dataset_2025_csv"), Path("../data/steam_dataset_2025_csv"),
    Path("Jacob/steam_dataset_2025_csv"),
]
DATA_DIR = next((p for p in CANDIDATES if (p / "applications.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("steam_dataset_2025_csv/ not found. Download it from Kaggle and put the "
                            "folder next to this notebook, or add its path to CANDIDATES.")
OUT_DIR = Path("clean")  # cleaned output (csv.gz), small enough to share
REVIEW_NROWS = None  # set e.g. 200_000 for a quick dev run
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# genres.csv has 154 rows because the same genre exists once per language, so map ids -> English.
GENRE_IDS = {
    "Action": [6, 8, 13, 14, 22, 64, 80, 120, 122, 125, 133, 136, 142, 145, 160],
    "Adventure": [1, 18, 26, 39, 42, 43, 54, 57, 61, 85, 94, 101, 108, 113, 128, 132, 150],
    "Casual": [29, 32, 34, 35, 52, 90, 92, 110, 127, 135],
    "Indie": [3, 7, 24, 31, 48, 72, 76, 96, 115, 126, 146],
    "RPG": [40, 51, 65, 69, 119, 124, 144],
    "Simulation": [10, 11, 17, 20, 37, 41, 67, 71, 87, 91, 99, 103, 109, 114, 137, 138, 141, 143, 147],
    "Strategy": [4, 5, 30, 46, 56, 70, 78, 84, 88, 102, 105, 112, 118, 139],
    "Sports": [60, 62, 77, 89, 97, 104, 111, 129, 140],
    "Racing": [15, 63, 66, 73, 74, 121, 149],
    "Massively Multiplayer": [2, 16, 19, 25, 55, 58, 59, 148],
    "Free To Play": [21, 28, 36, 44, 75, 93, 95, 117, 167],
    "Early Access": [9, 107],
    # software / content flags, mapped only so they can be dropped on purpose
    "Education": [38, 130], "Game Development": [86, 123], "Web Publishing": [27, 98],
    "Sexual Content": [12], "Episodic": [23], "Documentary": [33], "Animation & Modeling": [45],
    "Design & Illustration": [47], "Nudity": [49], "Software Training": [50], "Photo Editing": [53],
    "Audio Production": [68], "Gore": [79], "Accounting": [81], "Tutorial": [82],
    "Video Production": [83], "Violent": [106], "Utilities": [116], "Movie": [131],
    "360 Video": [134], "Short": [151],
}
ID_TO_GENRE = {i: g for g, ids in GENRE_IDS.items() for i in ids}

# Gameplay genres (Indie is NOT a genre, see is_indie) in "primary genre" priority order (niche first, so Action/Adventure don't
# swallow everything). This is a modeling knob: change the order and the target changes.
PRIORITY = ["RPG", "Strategy", "Simulation", "Sports", "Racing", "Massively Multiplayer",
            "Adventure", "Action", "Casual"]

In [ ]:
# Look at the files, and find the reviews column names (I resolve them from the header).
for p in sorted(DATA_DIR.glob("*.csv")):
    print(f"{p.name:28s} {p.stat().st_size / 1e6:8.1f} MB")

REVIEWS_CSV = DATA_DIR / "reviews.csv"
hdr = pd.read_csv(REVIEWS_CSV, nrows=0).columns.tolist()
print("\nreviews.csv columns:", hdr)

def pick(*cands):
    return next((c for c in cands if c in hdr), None)

COL = {
    "user": pick("author_steamid", "steamid", "author_id", "user_id"),
    "item": pick("appid", "app_id", "steam_appid"),
    "time": pick("timestamp_created", "review_timestamp", "timestamp", "created"),
    "voted_up": pick("voted_up", "recommended", "recommend"),
    "playtime_at_review": pick("playtime_at_review", "author_playtime_at_review"),
    "playtime_forever": pick("playtime_forever", "author_playtime_forever"),
    "votes_up": pick("votes_up"),
    "votes_funny": pick("votes_funny"),
    "steam_purchase": pick("steam_purchase"),
    "received_for_free": pick("received_for_free"),
    "early_access_review": pick("written_during_early_access"),
    "games_owned": pick("author_num_games_owned", "num_games_owned"),
    "author_reviews": pick("author_num_reviews", "num_reviews"),
}
print("\nresolved:", {k: v for k, v in COL.items() if v})
missing = [k for k in ("user", "item", "time") if COL[k] is None]
assert not missing, f"could not find a column for {missing}; edit COL above using the printed header"
pd.read_csv(REVIEWS_CSV, nrows=3, usecols=[c for c in COL.values() if c])

In [ ]:
# Games: applications of type "game" with a genre list from the join table
apps = pd.read_csv(
    DATA_DIR / "applications.csv",
    usecols=["appid", "name", "type", "is_free", "release_date", "metacritic_score",
             "recommendations_total", "mat_initial_price", "mat_currency"],
)
print(apps["type"].value_counts().to_dict())
apps = apps[apps["type"] == "game"].drop(columns="type")

# release_date has junk like 1969-12-31 and 9998-12-31 -> treat as missing
apps["release_date"] = pd.to_datetime(apps["release_date"], errors="coerce")
apps.loc[~apps["release_date"].between("1990-01-01", "2026-12-31"), "release_date"] = pd.NaT
apps["release_year"] = apps["release_date"].dt.year
# price is in cents; only trust USD rows, and drop absurd values
apps["price_usd"] = np.where(apps["mat_currency"] == "USD", apps["mat_initial_price"] / 100, np.nan)
apps.loc[apps["price_usd"] > 1000, "price_usd"] = np.nan

ag = pd.read_csv(DATA_DIR / "application_genres.csv")
ag["genre"] = ag["genre_id"].map(ID_TO_GENRE)
print("genre rows with an unmapped id:", ag["genre"].isna().sum(), "of", len(ag),
      "->", sorted(int(i) for i in ag.loc[ag["genre"].isna(), "genre_id"].unique()))
ag = ag.dropna(subset=["genre"]).drop_duplicates(["appid", "genre"])
genre_sets = ag.groupby("appid")["genre"].agg(set)

games = apps.merge(genre_sets.rename("genre_set"), left_on="appid", right_index=True, how="inner")
# genres = gameplay genres only. Indie is about who made the game, not how it plays, so it is
# kept as a separate yes/no flag (is_indie) instead of a genre.
games["genres"] = games["genre_set"].map(lambda s: [g for g in PRIORITY if g in s])
games["is_indie"] = games["genre_set"].map(lambda s: "Indie" in s)
games["is_f2p"] = games["genre_set"].map(lambda s: "Free To Play" in s) | games["is_free"].fillna(False).astype(bool)
games["is_early_access"] = games["genre_set"].map(lambda s: "Early Access" in s)
n_all = len(games)
# need at least one gameplay genre
games = games[games["genres"].map(len) > 0].copy()
games["primary_genre"] = games["genres"].str[0]
games["n_genres"] = games["genres"].map(len)
games = games.drop(columns=["genre_set", "mat_initial_price", "mat_currency", "is_free"])
print(f"games with genre data: {n_all:,} -> with a gameplay genre: {len(games):,}")
games.head()

In [ ]:
# Reviews: only the columns we need (skips the big review text), then standard names
use = {v: k for k, v in COL.items() if v}
rev = pd.read_csv(REVIEWS_CSV, usecols=list(use), nrows=REVIEW_NROWS).rename(columns=use)
rev["pos"] = np.arange(len(rev))  # original row order, used as a tie-break
print(f"{len(rev):,} reviews, {rev['user'].nunique():,} users, {rev['item'].nunique():,} games")

# time: unix seconds (or ms), otherwise a datetime string
t = rev["time"]
if pd.api.types.is_numeric_dtype(t):
    unit = "ms" if t.dropna().median() > 1e11 else "s"
    rev["date"] = pd.to_datetime(t, unit=unit, errors="coerce")
else:
    rev["date"] = pd.to_datetime(t, errors="coerce", utc=True).dt.tz_localize(None)
print("unparsed times:", rev["date"].isna().sum(), "| range:", rev["date"].min(), "to", rev["date"].max())

for c in ["voted_up", "steam_purchase", "received_for_free", "early_access_review"]:
    if c in rev:
        rev[c] = rev[c].astype(str).str.lower().isin(["true", "1", "1.0"])

In [ ]:
# THE key check first: how many reviews does each user have? Next-genre needs a history.
# (~1M reviews spread over 200k+ apps could mean most users have one or two.)
per_user_raw = rev.groupby("user").size()
print("reviews per user (raw, all games):")
print(per_user_raw.describe().round(2))
for k in (1, 2, 3, 5, 10):
    print(f"users with >= {k:2d} reviews: {(per_user_raw >= k).sum():>9,} ({(per_user_raw >= k).mean():.1%})")

In [ ]:
# Clean: dated, known game with a gameplay genre, one review per (user, game)
n0 = len(rev)
rev = rev.dropna(subset=["date"])
n1 = len(rev)
keep_game_cols = ["appid", "genres", "primary_genre", "n_genres", "price_usd", "release_date",
                  "release_year", "is_f2p", "is_early_access", "is_indie", "metacritic_score", "recommendations_total"]
rev = rev.merge(games[keep_game_cols].rename(columns={"appid": "item"}), on="item", how="inner")
n2 = len(rev)
rev = rev.sort_values(["user", "item", "date", "pos"]).drop_duplicates(["user", "item"])
n3 = len(rev)
rev = rev.sort_values(["user", "date", "pos"]).reset_index(drop=True)
print(f"raw {n0:,} -> dated {n1:,} -> game with gameplay genre {n2:,} -> one per user+game {n3:,}")

rev["seq_idx"] = rev.groupby("user").cumcount()
rev["n_events"] = rev.groupby("user")["item"].transform("size")
rev["days_since_prev"] = rev.groupby("user")["date"].diff().dt.total_seconds() / 86400
rev["days_since_release"] = (rev["date"] - rev["release_date"]).dt.days
for c in ("playtime_at_review", "playtime_forever"):
    if c in rev:
        rev[c + "_h"] = rev[c] / 60  # minutes -> hours

per_user = rev.groupby("user").size()
print("\nusable reviews per user:")
print(per_user.describe().round(2))
for k in (2, 3, 5, 10):
    print(f"users with >= {k:2d} usable reviews: {(per_user >= k).sum():>9,}")
rev.head()

In [ ]:
# Genre popularity and the global-mode baseline
print("date range:", rev["date"].min().date(), "to", rev["date"].max().date())
primary_counts = rev["primary_genre"].value_counts()
any_counts = Counter(g for gs in rev["genres"] for g in gs)
cmp = pd.DataFrame({
    "primary_share": primary_counts / primary_counts.sum(),
    "any_genre_share": pd.Series(any_counts) / len(rev),
}).sort_values("primary_share", ascending=False).fillna(0)
print(f"global mode primary genre: {primary_counts.index[0]} ({primary_counts.iloc[0] / len(rev):.1%} of reviews)")
display(cmp)

game_share = games["primary_genre"].value_counts(normalize=True).rename("share_of_games")
display(pd.concat([cmp["primary_share"], game_share], axis=1).round(3))  # reviews vs catalog mix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cmp["primary_share"][::-1].plot.barh(ax=axes[0], title="primary genre share of reviews")
rev.groupby(rev["date"].dt.to_period("Q")).size().plot(ax=axes[1], title="reviews per quarter")
plt.tight_layout()
plt.show()

In [ ]:
# History lengths (usable) and time-order quality
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
per_user.clip(upper=20).plot.hist(bins=20, ax=axes[0], title="usable reviews per user (clipped at 20)")
steps = rev[rev["seq_idx"] > 0]
steps["days_since_prev"].clip(upper=730).plot.hist(bins=50, ax=axes[1], title="days between consecutive reviews (clipped at 730)")
plt.tight_layout()
plt.show()

print(f"steps with the exact same timestamp as the previous review: {(steps['days_since_prev'] == 0).mean():.2%}")
print(f"steps within the same day: {(steps['days_since_prev'] < 1).mean():.1%}")

In [ ]:
# Genre structure: how many genres per game, and do users repeat genres?
print(games["n_genres"].value_counts().sort_index().to_string())
print(f"games flagged Indie: {games['is_indie'].mean():.1%}")

nxt = rev.groupby("user")["primary_genre"].shift(-1)
pairs = rev.assign(next_genre=nxt).dropna(subset=["next_genre"]).copy()
pairs["switch"] = pairs["primary_genre"] != pairs["next_genre"]
print(f"consecutive pairs: {len(pairs):,}; next review has the same primary genre: {1 - pairs['switch'].mean():.1%}")

order = list(primary_counts.index)
tm = pd.crosstab(pairs["primary_genre"], pairs["next_genre"]).reindex(index=order, columns=order).fillna(0)
tm = tm.div(tm.sum(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(tm.values, cmap="Blues")
ax.set_xticks(range(len(order)), order, rotation=90)
ax.set_yticks(range(len(order)), order)
ax.set_xlabel("next genre")
ax.set_ylabel("current genre")
ax.set_title("P(next | current), primary genre")
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
# Baseline sanity check on each user's last review (history = everything before it)
seq = rev.groupby("user")["primary_genre"].apply(list)
seq = seq[seq.map(len) >= 3]
hist = seq.map(lambda s: s[:-1])
target = seq.map(lambda s: s[-1])
if len(seq):
    global_mode = Counter(g for s in hist for g in s).most_common(1)[0][0]
    user_mode = hist.map(lambda s: Counter(s).most_common(1)[0][0])
    last_seen = hist.map(lambda s: s[-1])
    print(f"users with >= 3 reviews: {len(seq):,}")
    print(f"global mode ({global_mode}): {(target == global_mode).mean():.3f}")
    print(f"repeat last genre:          {(target == last_seen).mean():.3f}")
    print(f"user's own mode:            {(target == user_mode).mean():.3f}")
else:
    print("no user has 3 usable reviews: see the history-length check above")

In [ ]:
# Candidate parameters beyond time order: does the current review's context predict a switch next?
f = pairs.copy()
f["gap"] = pd.cut(f["days_since_prev"], [-1, 0, 1, 7, 30, 180, 1e6], labels=["same time", "<=1d", "<=1w", "<=1m", "<=6m", ">6m"])
f["price"] = pd.cut(f["price_usd"], [-0.01, 0, 10, 20, 40, 1000], labels=["free", "<=10", "<=20", "<=40", ">40"])
f["age_at_review"] = pd.cut(f["days_since_release"], [-1e6, 30, 365, 1095, 1e6], labels=["<=1m", "<=1y", "<=3y", ">3y"])
f["history_len"] = pd.cut(f["seq_idx"], [-1, 0, 2, 5, 1e9], labels=["1st", "2-3", "4-6", "7+"])
f["metacritic"] = pd.cut(f["metacritic_score"], [0, 60, 75, 90, 100], labels=["<=60", "<=75", "<=90", ">90"])
if "playtime_at_review_h" in f:
    f["hours_at_review"] = pd.cut(f["playtime_at_review_h"], [-1, 2, 10, 50, 1e9], labels=["<=2h", "<=10h", "<=50h", ">50h"])
if "games_owned" in f:
    f["library_size"] = pd.cut(f["games_owned"], [-1, 10, 50, 200, 1e9], labels=["<=10", "<=50", "<=200", ">200"])
cands = ["voted_up", "steam_purchase", "received_for_free", "early_access_review", "is_f2p", "is_early_access", "is_indie",
         "gap", "price", "age_at_review", "history_len", "metacritic", "hours_at_review", "library_size"]
base = f["switch"].mean()
print(f"overall switch rate: {base:.3f}")
for col in [c for c in cands if c in f]:
    t = f.groupby(col, observed=True)["switch"].agg(["mean", "size"]).rename(columns={"mean": "switch_rate"})
    print(f"\n{col}")
    print(t.round(3))

In [ ]:
# Does the current genre or the user's activity level change how predictable the next genre is?
by_genre = pairs.groupby("primary_genre")["switch"].agg(["mean", "size"]).rename(columns={"mean": "switch_rate"})
print(by_genre.sort_values("switch_rate").round(3))

# do users who review more repeat genres more?  (concentration of a user's history in one genre)
conc = rev.groupby("user")["primary_genre"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
conc_df = pd.DataFrame({"n": per_user, "top_genre_share": conc})
conc_df = conc_df[conc_df["n"] >= 3]
conc_df["n_bucket"] = pd.cut(conc_df["n"], [2, 3, 5, 10, 1e9], labels=["3", "4-5", "6-10", "11+"])
print("\ntop-genre share of a user's history, by history length")
print(conc_df.groupby("n_bucket", observed=True)["top_genre_share"].agg(["mean", "size"]).round(3))

In [ ]:
# Save cleaned tables as plain csv.gz (no pickle) for prototype_models.py
OUT_DIR.mkdir(exist_ok=True)
keep = ["user", "item", "date", "seq_idx", "n_events", "primary_genre", "genres", "voted_up",
        "price_usd", "release_year", "is_f2p", "is_early_access", "is_indie", "metacritic_score",
        "days_since_prev", "days_since_release", "playtime_at_review_h", "playtime_forever_h",
        "votes_up", "votes_funny", "steam_purchase", "received_for_free", "games_owned", "author_reviews"]
keep = [c for c in keep if c in rev.columns]
out = rev[keep].rename(columns={"user": "user_id", "item": "item_id"}).copy()
out["genres"] = out["genres"].map("|".join)  # list -> "RPG|Action"
out.to_csv(OUT_DIR / "clean_reviews.csv.gz", index=False)
games_out = games.copy()
games_out["genres"] = games_out["genres"].map("|".join)
games_out.to_csv(OUT_DIR / "clean_games.csv.gz", index=False)
for f in sorted(OUT_DIR.glob("*.csv.gz")):
    mb = f.stat().st_size / 1e6
    print(f"{f.name}: {mb:.1f} MB" + ("  <- over the 100 MB GitHub limit, do not commit" if mb > 95 else ""))